In [30]:
# !pip install langchain

In [31]:

import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb

load_dotenv()

# Embedding Manager
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
    
    def generate_embeddings(self, texts):
        return self.model.encode(texts)

# Retriever
class SimpleRetriever:
    def __init__(self, embedding_manager):
        self.client = chromadb.PersistentClient(path="C:\\Users\\USER\\Desktop\\pinecone-demo\\data\\vectorstore")
        self.collection = self.client.get_or_create_collection("documents")
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query, top_k=3):
        query_embedding = self.embedding_manager.generate_embeddings([query])[0].tolist()
        results = self.collection.query(query_embeddings=[query_embedding], n_results=top_k)
        if results["documents"] and results["documents"][0]:
            return [{"document": doc} for doc in results["documents"][0]]
        return []

embedding_manager = EmbeddingManager()
retriever = SimpleRetriever(embedding_manager)
print("Setup complete")

Setup complete


In [32]:
# Multi-Agent System with Tools
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Correct imports for LangChain 0.1+
from langchain.agents import AgentExecutor, create_react_agent

# Security Toolkit
class SecurityToolkit:
    def __init__(self, retriever):
        self.retriever = retriever
    
    def search_vector_db(self, query: str) -> str:
        """Search internal security documentation and knowledge base"""
        results = self.retriever.retrieve(query, top_k=3)
        if not results:
            return "No relevant documents found."
        return "\n".join([r['document'][:200] for r in results])
    
    def analyze_threat(self, description: str) -> str:
        """Analyze security threats and determine severity level"""
        threat_keywords = ['sql injection', 'xss', 'ddos', 'malware', 'phishing', 'ransomware']
        severity = sum(1 for k in threat_keywords if k in description.lower())
        if severity > 2:
            return "Threat Level: CRITICAL"
        elif severity > 1:
            return "Threat Level: HIGH"
        elif severity == 1:
            return "Threat Level: MEDIUM"
        return "Threat Level: LOW"
    
    def check_compliance(self, requirement: str) -> str:
        """Check applicable compliance frameworks (GDPR, HIPAA, PCI, SOC2)"""
        frameworks = {
            'gdpr': 'GDPR - EU Data Protection',
            'hipaa': 'HIPAA - Healthcare Security',
            'pci': 'PCI-DSS - Payment Card Security',
            'soc2': 'SOC2 - Service Organization Control'
        }
        matches = [v for k, v in frameworks.items() if k in requirement.lower()]
        return f"Applicable frameworks: {', '.join(matches)}" if matches else "General security best practices apply"

toolkit = SecurityToolkit(retriever)

# Define Tools using StructuredTool
tools = [
    StructuredTool.from_function(
        func=toolkit.search_vector_db,
        name="VectorDBSearch",
        description="Search internal security documentation and knowledge base"
    ),
    StructuredTool.from_function(
        func=toolkit.analyze_threat,
        name="ThreatAnalysis",
        description="Analyze security threats and determine severity level"
    ),
    StructuredTool.from_function(
        func=toolkit.check_compliance,
        name="ComplianceCheck",
        description="Check applicable compliance frameworks (GDPR, HIPAA, PCI, SOC2)"
    )
]

# ReAct Agent Prompt
react_prompt = PromptTemplate.from_template("""You are a cybersecurity expert agent. Use tools to answer questions accurately.

Tools available:
{tools}

Tool Names: {tool_names}

Use this format:
Question: the input question
Thought: think about what to do
Action: the tool name
Action Input: the input to the tool
Observation: the result
... (repeat as needed)
Thought: I now know the answer
Final Answer: the final answer

Question: {input}
{agent_scratchpad}""")

# Create Agent
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True, 
    handle_parsing_errors=True,
    max_iterations=5
)
print("Agent ready")

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Users\USER\Desktop\pinecone-demo\.venv\Lib\site-packages\langchain\agents\__init__.py)

In [ ]:
# Test the Multi-Agent System
questions = [
    "What is RAG and how do I implement it securely?",
    "Analyze this threat: User input allows SQL injection in login form",
    "What GDPR requirements apply to storing user embeddings?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print('='*60)
    result = agent_executor.invoke({"input": q})
    print(f"A: {result['output']}")